# قراءة ملف CSV : اولا 


أول خطوة سويتها إني قرأت ملف articles.csv 
خزنت id والعنوان والمحتوى في قوائم عشان أستخدمهم بعدين

In [9]:
import csv

ids = []
titles = []
contents = []

with open("articles.csv", "r", encoding="utf-8") as file:
    reader = csv.reader(file)
    header = next(reader)
    
    for row in reader:
        ids.append(row[0])
        titles.append(row[1])
        contents.append(row[2])

 # ثانينا : تنظيف النص
بعد ما قرأت البيانات، نظفت النصوص.
حولت الكلام لحروف صغيرة، بعدت علامات الترقيم، بعدت الأرقام، وبعدين قسمت النص إلى كلمات باستخدام split().

In [10]:
import string

def clean_text(text):
    text = text.lower()
    
    for p in string.punctuation:
        text = text.replace(p, "")
    
    cleaned = ""
    for ch in text:
        if not ch.isdigit():
            cleaned += ch
    
    words = cleaned.split()
    return words


cleaned_articles = []

for text in contents:
    cleaned_articles.append(clean_text(text))

#  بناء Global Vocabulary : ثالثا 

بعد التنظيف، جمعت كل الكلمات المختلفة من كل المقالات.
سويت قائمة فيها كل الكلمات الفريدة، وهذه هي الـ Global Bag of Words.

In [11]:
global_vocab = []

for article in cleaned_articles:
    for word in article:
        if word not in global_vocab:
            global_vocab.append(word)

#   تحويل المقالات إلى Vectors : رابعا

لكل مقال بنيت vector بنفس طول الـ vocabulary
إذا الكلمة موجودة في المقال أحط 1، وإذا مش موجودة أحط 0

In [12]:
import numpy as np

def article_to_vector(article_words):
    vector = []
    for word in global_vocab:
        if word in article_words:
            vector.append(1)
        else:
            vector.append(0)
    return np.array(vector)


article_vectors = []

for article in cleaned_articles:
    vec = article_to_vector(article)
    article_vectors.append(vec)

article_vectors = np.array(article_vectors)

# حساب Cosine Similarity : خامسا

بعد ما صار عندي متجه لكل مقال، حسبت Cosine Similarity بينهم باستخدام numpy.
استخدمت:
dot product
طول كل متجه
المعادلة المطلوبة

In [13]:
dot_product = np.dot(article_vectors, article_vectors.T)

norms = np.linalg.norm(article_vectors, axis=1)

similarity_matrix = dot_product / (norms[:, None] * norms[None, :])

 # حفظ النتيجة في ملف PKL : سادسا
بعد ما حسبت مصفوفة التشابه، خزنتها في ملف باسم similarities.pkl باستخدام pickle عشان أقدر أستخدمها بعدين بدون إعادة الحساب

In [14]:
import pickle

with open("similarities.pkl", "wb") as f:
    pickle.dump(similarity_matrix, f)

# سابعا : إيجاد أكثر 3 مقالات تشابهًا
سويت دالة تستقبل article_id وترجع لي أكثر 3 مقالات تشابهًا معه (بدون نفسه).
رتبت النتائج من الأعلى للأقل باستخدام argsort.

In [15]:
def find_top_similar(article_id):
    
    if str(article_id) not in ids:
        return "Article not found"
    
    index = ids.index(str(article_id))
    
    sims = similarity_matrix[index]
    
    sorted_indices = np.argsort(-sims)
    
    results = []
    
    for i in sorted_indices:
        if i != index:
            results.append(titles[i])
        if len(results) == 3:
            break
    
    return results


print(find_top_similar(1))

['The Growth of Internet of Things', 'Introduction to Cloud Computing', 'Database Management Fundamentals']
